# Estimating Population and Demographic Characteristics in Nigeria Using Google Colab

## Seven-Chapter Student Notebook

Choose a Nigerian State/FCT and LGA/Area Council, then progressively analyse mapped settlements, population, sex composition, children under 1 and under 5, detected buildings, and health facilities.

### Where your work is saved
This notebook mounts **your own Google Drive**. Final work is saved permanently under:

`MyDrive/Nigeria_Population_Demography_Course/Student_Work/<State>/<LGA>/`

Temporary processing files may still use `/content`, but your selected boundary and final results are copied to Google Drive so they remain available after the Colab runtime ends.

**Scientific rules:** population values are modelled estimates; GRID3 settlement extents are mapped extents rather than official community boundaries; building counts are detected footprints rather than households; facility presence does not prove operational status; under-1 is already included in under-5.

Run the notebook from top to bottom.



# Chapter 1 — Choose and map your study LGA

Select a State/FCT and LGA/Area Council from the validated SALB administrative lookup. Stable IDs are `adm1cd` and `adm2cd`.


In [ ]:
# Cell 1 — Install tools, mount Google Drive, and create your personal course folder
!pip -q install geopandas pyogrio shapely pyproj rasterio exactextract requests pyarrow fiona s2sphere ipywidgets folium

from pathlib import Path
import json, re, subprocess, sys, requests
import pandas as pd
import geopandas as gpd
import numpy as np
import folium
import ipywidgets as widgets
from IPython.display import display
from google.colab import drive

# Mount the student's own Google Drive.
drive.mount("/content/drive")

REPO="zubairgis/nigeria-population-demography-course"
RAW=f"https://raw.githubusercontent.com/{REPO}/main"

# Temporary processing area. This may disappear when the Colab runtime resets.
TEMP_ROOT=Path("/content/nigeria_demography_course")
TEMP_ROOT.mkdir(parents=True,exist_ok=True)

# Persistent student area. Files here remain in the student's Google Drive.
DRIVE_ROOT=Path("/content/drive/MyDrive/Nigeria_Population_Demography_Course/Student_Work")
DRIVE_ROOT.mkdir(parents=True,exist_ok=True)

states=pd.read_csv(f"{RAW}/data/lookups/admin_states_source.csv",dtype=str)
lgas=pd.read_csv(f"{RAW}/data/lookups/admin_lgas_source.csv",dtype=str)

assert len(states)==37 and len(lgas)==774
print("Google Drive mounted successfully.")
print("Persistent course folder:", DRIVE_ROOT)
print("State/FCT:",len(states)," LGA/Area Council:",len(lgas))



In [ ]:
# Cell 2 — Choose State/FCT and LGA
state_dropdown=widgets.Dropdown(options=sorted(states.adm1nm.unique()),value="Bayelsa",description="State/FCT:")
lga_dropdown=widgets.Dropdown(description="LGA:")
def refresh(*_):
    opts=sorted(lgas.loc[lgas.adm1nm.eq(state_dropdown.value),"adm2nm"].unique())
    lga_dropdown.options=opts
    lga_dropdown.value="Sagbama" if state_dropdown.value=="Bayelsa" and "Sagbama" in opts else opts[0]
state_dropdown.observe(refresh,names="value"); refresh()
display(widgets.VBox([state_dropdown,lga_dropdown]))
print("Choose above, then run Cell 3.")


In [ ]:
# Cell 3 — Load selected SALB boundary, create your personal work folder, and map it
SELECTED_STATE=state_dropdown.value
SELECTED_LGA=lga_dropdown.value

STATE_ID=states.loc[states.adm1nm.eq(SELECTED_STATE),"adm1cd"].iloc[0]
safe=lambda s: re.sub(r"[^A-Za-z0-9]+","_",str(s)).strip("_")

# Persistent folder for this student's selected study area.
STUDENT_WORK_DIR=DRIVE_ROOT/safe(SELECTED_STATE)/safe(SELECTED_LGA)
STUDENT_WORK_DIR.mkdir(parents=True,exist_ok=True)

# Temporary processing folder for this run.
WORK=TEMP_ROOT/safe(SELECTED_STATE)/safe(SELECTED_LGA)
WORK.mkdir(parents=True,exist_ok=True)

url=f"{RAW}/data/boundaries/source/BNDA_NGA_2000-01-01_lastupdate/by_state/{STATE_ID}_{safe(SELECTED_STATE)}.geojson"
state_gdf=gpd.read_file(url).to_crs(4326)
selected=state_gdf[state_gdf.adm2nm.astype(str).eq(SELECTED_LGA)].copy()

if len(selected)!=1:
    raise ValueError("Selected LGA was not uniquely resolved.")

LGA_ID=str(selected.iloc[0].adm2cd)

# Save the boundary both temporarily and permanently.
LGA_FILE=WORK/f"{LGA_ID}.geojson"
LGA_DRIVE_FILE=STUDENT_WORK_DIR/f"{LGA_ID}_boundary.geojson"
selected.to_file(LGA_FILE,driver="GeoJSON")
selected.to_file(LGA_DRIVE_FILE,driver="GeoJSON")

BBOX=tuple(float(x) for x in selected.total_bounds)
geom=selected.geometry.iloc[0]
center=[geom.centroid.y,geom.centroid.x]

# Tile-free map: avoids OpenStreetMap 403/blocking in shared Colab environments.
m=folium.Map(location=center,zoom_start=9,tiles=None,control_scale=True)
folium.GeoJson(
    selected.__geo_interface__,
    name="Selected LGA",
    style_function=lambda feature: {
        "fillColor": "#d9ead3",
        "color": "#1b5e20",
        "weight": 2,
        "fillOpacity": 0.45
    },
    tooltip=folium.Tooltip(f"{SELECTED_STATE} — {SELECTED_LGA}")
).add_to(m)

# Fit the map exactly to the selected LGA.
minx,miny,maxx,maxy=BBOX
m.fit_bounds([[miny,minx],[maxy,maxx]])

# White background so no external basemap tiles are required.
m.get_root().html.add_child(
    folium.Element("<style>.leaflet-container{background:#ffffff !important;}</style>")
)

display(m)

print("State ID:",STATE_ID)
print("LGA ID:",LGA_ID)
print("Persistent student folder:",STUDENT_WORK_DIR)
print("Boundary saved to Google Drive:",LGA_DRIVE_FILE)




# Chapter 2 — Discover settlements and settlement names

Use GRID3 NGA Settlement Extents v3.1 and GRID3 Settlement Names. Zero-name and multiple-name cases remain explicit rather than being forced.


In [ ]:
# Cell 4 — Retrieve and prepare settlements
REPO_DIR=WORK/"repo"
if REPO_DIR.exists(): subprocess.run(["rm","-rf",str(REPO_DIR)],check=True)
subprocess.run(["git","clone","--depth","1",f"https://github.com/{REPO}.git",str(REPO_DIR)],check=True)
VEC=WORK/"vectors"; VEC.mkdir(exist_ok=True); bbox=[str(v) for v in BBOX]
for ds in ["settlement_extents","settlement_names"]:
    subprocess.run([sys.executable,str(REPO_DIR/"scripts/01_download_sources.py"),"--dataset",ds,"--output",str(VEC),"--bbox",*bbox],check=True)
SETT=WORK/"settlement_components.gpkg"
subprocess.run([sys.executable,str(REPO_DIR/"scripts/04_prepare_settlements.py"),"--extents",str(VEC/"settlement_extents.geojson"),"--names",str(VEC/"settlement_names.geojson"),"--lga",str(LGA_FILE),"--lga-id",LGA_ID,"--out",str(SETT)],check=True)
settlements=gpd.read_file(SETT).to_crs(4326)
print("Settlement components:",len(settlements)); display(settlements.name_match_status.value_counts())


# Chapter 3 — Estimate total population

WorldPop 2025 R2025A constrained 100 m rasters are counts of estimated people per grid square. Aggregation uses fractional pixel overlap.


In [ ]:
# Cell 5 — Download six WorldPop layers and estimate population
WP=WORK/"worldpop"; WP.mkdir(exist_ok=True)
subprocess.run([sys.executable,str(REPO_DIR/"scripts/01_download_sources.py"),"--dataset","worldpop_core","--output",str(WP)],check=True)
LGA_POP=WORK/"lga_population.csv"
subprocess.run([sys.executable,str(REPO_DIR/"scripts/03_prepare_population.py"),"--dir",str(WP),"--zones",str(LGA_FILE),"--id-field","adm2cd","--out",str(LGA_POP)],check=True)
SETT_POP=WORK/"settlement_population.csv"
subprocess.run([sys.executable,str(REPO_DIR/"scripts/03_prepare_population.py"),"--dir",str(WP),"--zones",str(SETT),"--id-field","settlement_component_id","--out",str(SETT_POP)],check=True)
lga_pop=pd.read_csv(LGA_POP); sett_pop=pd.read_csv(SETT_POP)
display(lga_pop)
inside=float(sett_pop.population_total_est.sum()); total=float(lga_pop.loc[0,"population_total_est"])
print(f"Inside mapped settlements: {inside:,.0f} ({100*inside/total:.2f}%)")
print(f"Outside mapped settlements: {total-inside:,.0f}")


# Chapter 4 — Describe male and female population

Total population is calculated from compatible male and female WorldPop layers.


In [ ]:
# Cell 6 — Male/female population
sex=pd.DataFrame({"group":["Male","Female"],"estimate":[float(lga_pop.loc[0,"population_male_est"]),float(lga_pop.loc[0,"population_female_est"])]})
display(sex)
ax=sex.plot.bar(x="group",y="estimate",legend=False,title=f"{SELECTED_LGA}: modelled male and female population")
ax.set_ylabel("Estimated people")


# Chapter 5 — Identify children under 1 and under 5

Under-1 = male age 0 + female age 0. Under-5 = age 0 plus ages 1–4. Never add under-1 again to under-5.


In [ ]:
# Cell 7 — Young-child estimates
u1=float(lga_pop.loc[0,"population_u1_est"]); u5=float(lga_pop.loc[0,"population_u5_est"]); total=float(lga_pop.loc[0,"population_total_est"])
assert 0<=u1<=u5<=total
display(pd.DataFrame({"indicator":["Under 1","Under 5"],"estimate":[u1,u5],"percent_total":[100*u1/total,100*u5/total]}).round(2))


# Chapter 6 — Count detected buildings and locate health facilities

Google Open Buildings V3 provides detected building footprints, not households. Health-facility records are spatially assigned after coordinate and duplicate checks.


In [ ]:
# Cell 8 — Health facilities and detected buildings
subprocess.run([sys.executable,str(REPO_DIR/"scripts/01_download_sources.py"),"--dataset","health_v2","--output",str(VEC),"--bbox",*bbox],check=True)
HEALTH=WORK/"health.gpkg"
subprocess.run([sys.executable,str(REPO_DIR/"scripts/06_prepare_health_facilities.py"),"--facilities",str(VEC/"health_v2.geojson"),"--lga",str(LGA_FILE),"--out",str(HEALTH)],check=True)
health=gpd.read_file(HEALTH).to_crs(4326)
OB=WORK/"open_buildings.parquet"; OBS=WORK/"open_buildings_summary.json"
subprocess.run([sys.executable,str(REPO_DIR/"scripts/05_download_open_buildings.py"),"--aoi",str(LGA_FILE),"--out",str(OB),"--summary",str(OBS),"--precision","90"],check=True)
BC=WORK/"building_counts.csv"; BAS=WORK/"building_allocation.json"
subprocess.run([sys.executable,str(REPO_DIR/"scripts/05_count_buildings.py"),"--buildings",str(OB),"--zones",str(SETT),"--zone-id","settlement_component_id","--out",str(BC),"--summary",str(BAS)],check=True)
ob=json.loads(OBS.read_text()); alloc=json.loads(BAS.read_text()); building_counts=pd.read_csv(BC)
print("Health facilities:",len(health)); print("Detected building footprints:",ob["detected_footprints_output"])
print("Allocated to settlements:",alloc["allocated_to_one_component"])


# Chapter 7 — Reconcile, interpret and export

Build final LGA and settlement summaries, run consistency checks, and export CSV/GeoJSON files.


In [ ]:
# Cell 9 — Final tables and checks
sett_table=settlements.drop(columns="geometry").merge(sett_pop,on="settlement_component_id",how="left",validate="one_to_one").merge(building_counts,on="settlement_component_id",how="left",validate="one_to_one")
sett_table["detected_building_count"]=sett_table["detected_building_count"].fillna(0).astype(int)
metrics=["population_total_est","population_male_est","population_female_est","population_u1_est","population_u5_est"]
recon=[]
for x in metrics:
    lv=float(lga_pop.loc[0,x]); iv=float(sett_pop[x].sum())
    recon.append({"metric":x,"lga_estimate":lv,"inside_mapped_settlements":iv,"outside_mapped_settlements":lv-iv,"coverage_pct":100*iv/lv})
reconciliation=pd.DataFrame(recon)
assert abs(float(lga_pop.loc[0,"population_male_est"])+float(lga_pop.loc[0,"population_female_est"])-float(lga_pop.loc[0,"population_total_est"]))<0.1
display(reconciliation.round(2))
display(sett_table[["settlement_name","name_match_status","detected_building_count","population_total_est","population_u1_est","population_u5_est"]].sort_values("population_total_est",ascending=False).head(20))


In [ ]:
# Cell 10 — Save all final results permanently in your Google Drive
EXPORT=STUDENT_WORK_DIR/"Final_Results"
EXPORT.mkdir(parents=True,exist_ok=True)

# Tables
lga_pop.to_csv(EXPORT/"lga_population_summary.csv",index=False)
sett_table.to_csv(EXPORT/"settlement_summary.csv",index=False)
reconciliation.to_csv(EXPORT/"population_reconciliation.csv",index=False)
health.drop(columns="geometry").to_csv(EXPORT/"health_facilities.csv",index=False)

# Spatial layers
selected.to_file(EXPORT/"selected_lga_boundary.geojson",driver="GeoJSON")
settlements.to_file(EXPORT/"settlement_components.geojson",driver="GeoJSON")

# Save a simple study-area map as HTML.
# Tile-free final map: no OpenStreetMap or other external tile server is required.
FINAL_MAP=folium.Map(location=center,zoom_start=9,tiles=None,control_scale=True)
FINAL_MAP.get_root().html.add_child(
    folium.Element("<style>.leaflet-container{background:#ffffff !important;}</style>")
)
folium.GeoJson(
    selected.__geo_interface__,
    name="Selected LGA",
    style_function=lambda feature: {
        "fillColor": "#d9ead3",
        "color": "#1b5e20",
        "weight": 2,
        "fillOpacity": 0.35
    }
).add_to(FINAL_MAP)
minx,miny,maxx,maxy=BBOX
FINAL_MAP.fit_bounds([[miny,minx],[maxy,maxx]])
if len(health)>0:
    for _,r in health.iterrows():
        folium.CircleMarker(
            location=[r.geometry.y,r.geometry.x],
            radius=3,
            tooltip=str(r.get("facility_name","Health facility"))
        ).add_to(FINAL_MAP)
FINAL_MAP.save(str(EXPORT/"study_area_map.html"))

# Create a ZIP inside Google Drive.
ZIP_DRIVE=STUDENT_WORK_DIR/f"{safe(SELECTED_STATE)}_{safe(SELECTED_LGA)}_demography_results.zip"
subprocess.run(["zip","-qr",str(ZIP_DRIVE),"."],cwd=EXPORT,check=True)

print("All final work has been saved permanently in Google Drive:")
print(EXPORT)
print("ZIP:",ZIP_DRIVE)

# Optional: also download the ZIP to the student's computer.
from google.colab import files
files.download(str(ZIP_DRIVE))




## Completed

You have selected an LGA, mapped settlements, estimated population and demographics, added building and health-facility context, reconciled the results, and exported the outputs.
